# High-speed native PMC post-configuration payloads

These cells use **recorded hardware evidence** by default. They do not program a board.
Use the configured Python (SCU35 PMC) kernel. Fresh hardware commands are shown in Markdown and must be invoked explicitly after configuring your exact SCU35 and recovery image.
Current PL timing uses AXI Timer; native PMC timing uses `mcycle`. JTAG/UART are outside operation timing.

Build with `python lab.py native`. Run explicitly with `python lab.py hardware --mode native --pdi /path/to/native/build/pmc_independent.pdi` after hardware setup. The native loader is experimental and separate from stock PLM secure boot.

Firmware `mcycle` timing includes setup/DMA/control, excluding JTAG/preload. Logical messages repeat resident chunks. Streamed AES validates the final chunk and whole-message tag. Rates are payload throughput, not measured link or HyperRAM bandwidth. PUF/RNG remain supported in the shared runtime path.

In [1]:
from pathlib import Path
import hashlib, json, sys, time
import pandas as pd
# Works from the notebook directory, lab root, or parent repository.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for base in candidates for p in (base, base / "scu35") if (p / "lab.py").is_file()), None)
assert ROOT is not None, "Launch Jupyter inside this checkout or its scu35 directory"
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "host"))
from compare import parse_capture
from lab import config
capture = parse_capture(ROOT / "evidence/timer/uart.txt", "7d1bf77bc454dd1f")
rows = pd.DataFrame(capture["rows"])
print("Recorded board capture: 31 complete rows; current Python:", sys.version.split()[0])


Recorded board capture: 31 complete rows; current Python: 3.11.9


In [2]:
sys.path.insert(0, str(ROOT / "native"))
import analyze
native = analyze.validate(ROOT / "evidence/native/results.txt")
assert len(native["rows"]) == 66 and len(native["pq_rows"]) == 24
display(pd.DataFrame(native["rows"]))
display(pd.DataFrame(native["pq_rows"]))
print("Native frequency estimate:", native["clock_hz"], "Hz;", native["clock_reference"])
print("Fresh Python SHA3 8 KiB batches:")
display(pd.DataFrame(analyze.python_measure(1, 8192, 100)))

,kind,algorithm,size_bytes,sample,repeats,cycles,words,mean_us,MBps
0,1,SHA3-256,16,0,1000,282022,"[1178686292, 282022, 16, 1000, 0, 0, 379097050...",1.157501,13.822885
1,1,SHA3-256,16,1,1000,282022,"[1178686292, 282022, 16, 1000, 0, 0, 379097050...",1.157501,13.822885
2,1,SHA3-256,16,2,1000,282022,"[1178686292, 282022, 16, 1000, 0, 0, 379097050...",1.157501,13.822885
3,1,SHA3-256,256,0,1000,402022,"[1178686292, 402022, 256, 1000, 0, 0, 20211094...",1.650016,155.150029
4,1,SHA3-256,256,1,1000,402022,"[1178686292, 402022, 256, 1000, 0, 0, 20211094...",1.650016,155.150029
...,...,...,...,...,...,...,...,...,...
61,5,AES-GCM-ENC-STREAM,65536,1,200,13762819,"[1178686292, 13762819, 65536, 200, 0, 0, 23322...",282.433162,232.040740
62,5,AES-GCM-ENC-STREAM,65536,2,200,13762820,"[1178686292, 13762820, 65536, 200, 0, 0, 15790...",282.433183,232.040723
63,5,AES-GCM-ENC-STREAM,1048576,0,200,216946904,"[1178686292, 216946904, 1048576, 200, 0, 0, 72...",4452.067568,235.525626
64,5,AES-GCM-ENC-STREAM,1048576,1,200,216946905,"[1178686292, 216946905, 1048576, 200, 0, 0, 29...",4452.067588,235.525625


,kind,algorithm,case,sample,repeats,cycles,hash_cycles,hash_calls,status,mean_us,hash_path_mean_us,software_remainder_mean_us,hash_calls_per_verification
0,6,LMS-SHAKE256,valid,0,20,4795314,1478240,4000,0,984.068523,303.356455,680.712068,200
1,6,LMS-SHAKE256,valid,1,20,4795315,1478240,4000,0,984.068728,303.356455,680.712274,200
2,6,LMS-SHAKE256,valid,2,20,4795315,1478240,4000,0,984.068728,303.356455,680.712274,200
3,6,LMS-SHAKE256,bad_signature,0,20,4517433,1370540,3700,1,927.043280,281.254840,645.788440,185
4,6,LMS-SHAKE256,bad_signature,1,20,4517434,1370540,3700,1,927.043485,281.254840,645.788645,185
5,6,LMS-SHAKE256,bad_signature,2,20,4517435,1370542,3700,1,927.043691,281.255251,645.788440,185
6,6,LMS-SHAKE256,bad_message,0,20,4739793,1456700,3940,1,972.674802,298.936132,673.738671,197
7,6,LMS-SHAKE256,bad_message,1,20,4739795,1456702,3940,1,972.675213,298.936542,673.738671,197
8,6,LMS-SHAKE256,bad_message,2,20,4739798,1456702,3940,1,972.675828,298.936542,673.739286,197
9,6,LMS-SHAKE256,bad_key,0,20,4795314,1478240,4000,1,984.068523,303.356455,680.712068,200


Native frequency estimate: 243647362.36926833 Hz; Nominal 100 MHz PL reference; not frequency-metrology calibration
Fresh Python SHA3 8 KiB batches:


,mean_us,MBps,batch_ns,repeats
0,19.089,429.147677,1908900,100
1,18.708,437.887535,1870800,100
2,18.986,431.475824,1898600,100


## Fresh hardware: explicit step

Configure `config.json` first. Run `python lab.py hardware --pdi /path/to/build/demo.pdi` from the lab directory for the full runtime suite, or the native/boot command above for that path. Save logs, verify recovery and compare results before discussing speed. See [audit checkpoints](../docs/audit.md).